# 📈 12장 · 작년 이맘때처럼: 시계열 분석 (v3 실습 노트북)

『빅데이터분석 with 파이썬』 12장의 **05. 파이썬 실습**, **04절 예제의 파이썬 확인**, **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호와 코드는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 12-1, 12-5, 12-6, 12-7, 12-8, 12-9 |
| ☆ 선택 | 자습이나 심화 | 12-2, 12-3, 12-4 |

**데이터**
- 국제선 항공 승객: 1949~1960년 월별 144개월(Box–Jenkins의 Series G). GitHub `jbrownlee/Datasets`에서 바로 읽습니다.
- 한국 전력 수요(12-5~12-9): 한국전력거래소 "시간별 전국 전력수요량" 2021~2025년(출처 한국전력거래소, 공공데이터포털, 공공누리). 실습 12-5 앞의 셀이 책 저장소에서 파일을 받아 옵니다.
- 미니 프로젝트: 독일 Open Power System Data의 하루 단위 파일 2006~2017년 4,383일. GitHub `jenfly/opsd`에서 바로 읽습니다.

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 프로펫을 쓰는 실습(12-7, 12-8)과 미니 프로젝트 3·4단계는 몇십 초씩 걸립니다.

## 0 · 준비

그래프의 한글이 깨지지 않게 하는 **koreanize-matplotlib**, 예측 도구 **프로펫(Prophet)**, 나라별 공휴일 **holidays**를 설치합니다.

In [ ]:
!pip install -q koreanize-matplotlib prophet holidays
import koreanize_matplotlib, warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

## 05. 파이썬 실습: 비행기 승객에서 전력 수요까지

실습은 코랩(Colab)에서 노트북 "12장_시계열_실습_colab_v2.ipynb"를 위에서부터 차례로 실행합니다. 한글 글꼴 설정은 노트북 첫 셀에 있습니다. 항공 승객과 독일 전력은 인터넷 주소에서 바로 읽고, 한국 전력 수요 파일(kpx_시간별_전국_전력수요량_2021_2025.csv)은 책 저장소의 ch12/data 폴더에서 받아 코랩 파일 창에 올립니다.

> **[STEP] 실습 전에 미리 살펴보기**
>
> 데이터: 항공 승객 144개월(12-1~12-4), 한국 하루 전력 1,826일(12-5~12-9), 미니 프로젝트는 독일 하루 전력 4,383일입니다.
>
> 순서: 12-1~12-4는 01·04절의 개념을, 12-5~12-9는 03절의 분석을 따라 합니다. 실습마다 ①②③ 단계 코드 → 실행 결과 → 결과 그래프 → 결과 읽기 순서입니다.
>
> 상자: [분석 기법]과 [자주 하는 실수]는 시계열이 아닌 데이터(매출, 집값, 고객 기록)에도 그대로 쓰는 기법입니다.

**준비: 라이브러리 불러오기**

In [ ]:
import numpy as np, pandas as pd                     # 숫자 계산과 표
import matplotlib.pyplot as plt                      # 그래프
from statsmodels.tsa.seasonal import seasonal_decompose   # 시계열 분해
from statsmodels.tsa.stattools import adfuller            # 정상성 검정
from statsmodels.tsa.statespace.sarimax import SARIMAX    # (계절) ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing  # 홀트-윈터스
from prophet import Prophet                          # 달력을 읽는 예측 도구
from sklearn.ensemble import RandomForestRegressor   # 랜덤 포레스트
import holidays                                      # 나라별 공휴일 달력

코랩에 없는 것은 `!pip install prophet holidays`로 먼저 설치합니다.


## [파이썬 실습 12-1 ★] 항공 승객 불러오기와 이동평균

항공 승객 데이터를 날짜 인덱스로 읽고, 12개월 이동평균으로 추세를 구해 그래프로 확인하시오.

**① 날짜를 인덱스로 읽기**

In [ ]:
URL_AIR = ("https://raw.githubusercontent.com/jbrownlee/Datasets/"
           "master/airline-passengers.csv")
air = pd.read_csv(URL_AIR, parse_dates=["Month"], index_col="Month")
y = air["Passengers"].asfreq("MS").astype(float)   # 매달 첫날(MS) 간격
print(len(y), "개월:", y.index[0].strftime("%Y-%m"), "~", y.index[-1].strftime("%Y-%m"))

- `parse_dates`는 "1949-01" 같은 글자를 날짜로 바꾸고, `index_col`은 그 열을 인덱스로 씁니다.
- `asfreq("MS")`: "매달 첫날" 간격을 정해, 빠진 달이 있으면 빈칸으로 드러냅니다.

> **[분석 기법] 날짜를 인덱스로: 시간이 붙은 데이터의 첫 단추**
>
> 카드 결제, 센서 값, 웹 접속 기록처럼 시각이 붙은 데이터는 날짜 열을 **인덱스**(줄 이름)로 둡니다. 그러면 `df.loc["2025-10"]`처럼 기간으로 고르고, `index.month`, `index.dayofweek`로 달과 요일을 바로 꺼내며, 그래프 가로축도 저절로 날짜가 됩니다. `asfreq( )`로 간격을 정해 빠진 날을 드러내야 "어제", "지난주" 계산이 어긋나지 않습니다.


**② 채점 함수와 연도별 요약**

In [ ]:
def mae(a, b):                                      # 평균 절대 오차 (식 12-5)
    return round(float(np.mean(np.abs(np.asarray(a) - np.asarray(b)))), 1)
year_mean = y.groupby(y.index.year).mean()          # 해마다 평균
print("1949년 평균", round(year_mean[1949], 1), "→ 1960년 평균", round(year_mean[1960], 1))
print("해마다 가장 붐빈 달:", y.groupby(y.index.year).idxmax().dt.month.value_counts().to_dict())

- `mae( )`는 뒤의 모든 실습에서 쓸 채점 함수입니다. `groupby(y.index.year)`는 같은 해끼리 묶고, `idxmax( )`는 가장 큰 값의 날짜를 돌려줍니다.

**③ 무게 13개로 이동평균 구하기**

In [ ]:
k = np.r_[0.5, np.ones(11), 0.5] / 12             # 무게 13개, 양 끝은 절반 (합은 1)
trend = y.rolling(13, center=True).apply(lambda w: w @ k)  # 창을 밀며 무게와 내적
print("추세 1949-07:", round(trend["1949-07-01"], 1), " 1960-06:", round(trend["1960-06-01"], 1))

- `rolling(13, center=True)`: 값 13개를 덮는 창을 가운데에 맞추어 한 칸씩 밉니다.
- `lambda w: w @ k`: "창의 값 w와 무게 k를 내적한다"는 한 줄짜리 함수입니다(식 12-1). 무게가 모두 같다면 `rolling(12).mean( )`으로 충분합니다.

**④ 그래프로 확인하기**

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(y, color="#8EC5E8", label="승객 수")
plt.plot(trend, color="#E04B3C", label="12개월 이동평균(추세)")
plt.ylabel("승객(천 명)"); plt.legend(); plt.show()

- 그래프의 틀은 늘 같습니다. 판 만들기(`figure`) → 그리기(`plot`, 막대는 `시리즈.plot.bar( )`) → 이름 붙이기(`ylabel`, `legend`) → 보이기(`show`). 이 장의 그래프는 모두 이 틀에 색만 바꾼 것입니다.



**[결과 읽기]** 열두 해 사이 승객이 약 3.8배로 늘었고, 해마다 7·8월이 가장 붐볐습니다. 양 끝 여섯 달은 창이 모자라 추세가 비어 있습니다.


## [파이썬 실습 12-2 ☆] 계절 지수와 정상성 검정

스태츠모델의 분해 함수로 달별 계절 지수를 구하고, 로그와 차분을 거칠 때마다 ADF 검정의 p값을 구하시오.

**① 곱셈 분해로 계절 지수 구하기**

In [ ]:
dec = seasonal_decompose(y, model="multiplicative", period=12)  # 추세 × 계절 × 나머지
season = dec.seasonal.iloc[:12]                     # 계절 지수는 해마다 같으니 열두 달만
season.index = season.index.month                   # 날짜 대신 1~12월로
print("계절 지수 1월", round(season[1], 3), "/ 7월", round(season[7], 3), "/ 11월", round(season[11], 3))

- `model="multiplicative"`: 흔들림이 추세에 비례해 커지므로 곱하기로 나눕니다(1.2). `period=12`는 열두 달마다 되풀이된다는 뜻이고, 하루 데이터의 요일이면 7입니다.

**② 로그와 차분을 거치며 ADF 검정**

In [ ]:
ly = np.log(y)                                      # 로그: 흔들림의 크기를 고르게
for name, s in {"원자료": y, "로그": ly, "로그+1차 차분": ly.diff(),
                "로그+1차+계절 차분": ly.diff().diff(12)}.items():
    print(name, "ADF p값", round(adfuller(s.dropna(), autolag="AIC")[1], 4))

- `diff( )`는 1차 차분, `diff(12)`는 계절 차분입니다(식 12-2). 맨 앞의 빈칸은 `dropna( )`로 버리고, `adfuller( )[1]`이 p값입니다.

> **[잠깐만] ADF 검정과 p값**
>
> **ADF 검정**은 "이 시계열은 고르지 않다(정상성이 없다)"는 가정을 시험합니다. **p값**(5장)은 "그 가정이 맞다면 이런 데이터가 나올 확률"이라, 0.05보다 작으면 가정을 버리고 "고르다"고 봅니다. 어느 시계열이든 ARIMA를 쓰기 전에 차분이 몇 번 필요한지 이 검정으로 확인합니다.


**③ 계절 지수 그래프**

In [ ]:
season.plot.bar(color=["#E04B3C" if m in (7, 8) else "#8EC5E8" for m in season.index],
                figsize=(7, 2.6), rot=0)
plt.axhline(1, color="gray", lw=1)                  # 1보다 크면 그해 흐름보다 붐비는 달
plt.xlabel("달"); plt.ylabel("계절 지수"); plt.show()

**[결과 읽기]** p값이 0.05 아래로 내려간 것은 계절 차분까지 한 뒤였습니다. 이 순서(로그 → 1차 차분 → 계절 차분)가 4.4의 항공사 모델이 하는 일입니다.

> **[분석 기법] 로그 변환: 커질수록 커지는 흔들림 고르기**
>
> 값이 커질수록 흔들림도 커지면 로그를 씌웁니다. 로그는 곱하기를 더하기로 바꾸어 "몇 배"의 변화를 "몇 칸"의 변화로 만듭니다. 소득, 집값, 매출처럼 큰 값 몇 개가 꼬리를 길게 끄는 데이터도 `np.log( )`(0이 있으면 `np.log1p( )`)로 고르게 만든 뒤 분석합니다. 예측은 `np.exp( )`로 되돌리는 것을 잊지 않습니다.



## [파이썬 실습 12-3 ☆] 네 가지 기준선과 겨루기

1958년까지로 배우고 1959~1960년 24개월을 네 가지 기준선으로 예측해 MAE를 구하시오.

**① 시간 순서대로 나누기**

In [ ]:
train, test = y.loc[:"1958"], y.loc["1959":]       # 1958년까지 학습, 1959~1960년 시험
h = len(test)                                       # 내다볼 달 수
print("학습", len(train), "개월 / 시험", h, "개월")

- `y.loc[:"1958"]`: 처음부터 1958년 12월까지입니다. 날짜 인덱스라 끝 날짜까지 포함됩니다.

> **[자주 하는 실수] 시간이 있는 데이터를 무작위로 나누기**
>
> 8장의 `train_test_split( )`은 줄을 섞어 나눕니다. 시계열을 이렇게 나누면 1959년 3월을 맞히는 데 2월과 4월이 학습에 들어가 점수가 실제보다 좋게 나옵니다. 주문 기록, 센서, 환자 방문처럼 시각이 있는 데이터는 날짜로 자르거나 `shuffle=False`를 주어 앞쪽을 학습, 뒤쪽을 시험에 씁니다.


**② 네 기준선 만들기**

In [ ]:
slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)   # 첫 값과 끝 값을 이은 기울기
base = {"평균": np.repeat(train.mean(), h),                   # 지금까지의 평균
        "직전 값": np.repeat(train.iloc[-1], h),               # 마지막 값 그대로
        "추세(drift)": train.iloc[-1] + slope * np.arange(1, h + 1),  # 기울기대로 연장
        "작년 같은 달": np.tile(train.iloc[-12:].values, 2)}     # 1958년 열두 달을 두 번

- `np.repeat(값, h)`는 같은 값을 h번 늘어놓고, `np.tile(배열, 2)`는 배열을 통째로 두 번 잇습니다. `iloc[-12:]`는 학습 기간의 마지막 열두 달(1958년)입니다.

**③ 채점하고 그래프로 견주기**

In [ ]:
score = pd.Series({k: mae(test, f) for k, f in base.items()})
print(score.to_string())
score.plot.barh(color=["#8EC5E8"] * 3 + ["#E04B3C"], figsize=(6, 2.6))
plt.xlabel("평균 오차 MAE(천 명)"); plt.show()

**[결과 읽기]** 계절을 아는 "작년 같은 달"(71.2)이 이 데이터에서 모델이 넘어야 할 선입니다.

> **[분석 기법] 기준선 모델: "아무 생각 없이"도 이만큼은 맞힌다**
>
> 모델의 성적은 가장 단순한 방법과 견줄 때 뜻이 생깁니다. 집값이면 "평균 집값", 스팸 분류면 "모두 정상 메일", 시계열이면 "직전 값"이나 "작년 같은 때"가 기준선이고, 사이킷런의 `DummyRegressor`, `DummyClassifier`가 이 일을 합니다. 보고할 때는 "MAE 39.4"가 아니라 "기준선 71.2보다 45% 작다"처럼 말합니다.



## [파이썬 실습 12-4 ☆] 항공사 모델, 홀트-윈터스, 기준점 옮기기

항공사 모델과 홀트-윈터스로 1959~1960년을 예측하고, 1955~1960년을 해마다 그 전해까지로 배워 예측해 두 모델과 결합 예측을 견주시오.

**① 두 모델을 함수로 만들기**

In [ ]:
def airline(series):                                # 항공사 모델: 로그 + 계절 ARIMA
    return SARIMAX(np.log(series), order=(0, 1, 1),
                   seasonal_order=(0, 1, 1, 12)).fit(disp=False)
def holt(series):                                   # 홀트-윈터스: 추세 + 곱셈 계절
    return ExponentialSmoothing(series, trend="add", seasonal="mul",
                                seasonal_periods=12).fit()

- `order=(0, 1, 1)`은 1차 차분 한 번과 "직전 오차 하나"를, `seasonal_order`는 열두 달 단위로 같은 일을 합니다(4.4 [더 알아보기]). 함수로 만들어 두면 ③에서 기간만 바꿔 다시 부릅니다.

**② 한 번 맞추고 95% 예측 구간 재기**

In [ ]:
sar = airline(train)
p_sar = np.exp(sar.forecast(h))                     # 로그로 배웠으니 exp로 되돌린다
p_hw = holt(train).forecast(h)
print("항공사 모델 MAE", mae(test, p_sar), "/ 홀트-윈터스 MAE", mae(test, p_hw))
ci = np.exp(sar.get_forecast(h).conf_int(alpha=0.05))   # 95% 예측 구간(아래, 위)
inside = ((test >= ci.iloc[:, 0]) & (test <= ci.iloc[:, 1])).sum()
print("95% 구간 안에 든 달:", inside, "/", h)

- `get_forecast( ).conf_int(alpha=0.05)`: 예측마다 95% 구간의 아래·위 끝을 두 열로 줍니다(6.3).

**③ 기준점을 한 해씩 옮기며 시험하기**

In [ ]:
rows = []
for Y in range(1955, 1961):                         # 1955~1960년을 차례로 시험
    tr, te = y.loc[:str(Y - 1)], y.loc[str(Y)]      # 그 전해까지 학습, 그해 시험
    f_sn = tr.iloc[-12:].values                     # 작년 같은 달
    f_sar = np.exp(airline(tr).forecast(12)).values
    f_hw = holt(tr).forecast(12).values
    rows.append({"시험 해": Y, "작년 같은 달": mae(te, f_sn), "SARIMA": mae(te, f_sar),
                 "홀트-윈터스": mae(te, f_hw), "둘의 평균": mae(te, (f_sar + f_hw) / 2)})
roll = pd.DataFrame(rows).set_index("시험 해")
print("해마다 1등:", roll.iloc[:, :3].idxmin(axis=1).value_counts().to_dict())
print("여섯 해 평균:", roll.mean().round(1).to_dict())

> **[분석 기법] 기준점을 옮기며 여러 번 시험하기**
>
> 시험을 한 번만 하면 그해의 운이 성적을 정합니다. 기준점을 옮기며 "그 전까지로 배우고 다음을 맞히기"를 되풀이하면(**워크 포워드 검증**) 평균과 최악을 함께 볼 수 있습니다. 사이킷런의 `TimeSeriesSplit(n_splits=5)`이 이 나누기를 해 줍니다. 8장의 교차 검증과 같은 생각이지만 학습이 늘 시험보다 앞섭니다.


**④ 예측과 구간 그래프**

In [ ]:
plt.figure(figsize=(7, 3))
plt.fill_between(test.index, ci.iloc[:, 0], ci.iloc[:, 1],      # 아래·위 끝 사이를 칠한다
                 color="#8EC5E8", alpha=0.4, label="95% 구간")
plt.plot(test, color="black", label="실제")
plt.plot(p_sar, color="#E04B3C", label="항공사 모델")
plt.plot(p_hw, color="#4466AE", ls="--", label="홀트-윈터스")
plt.gcf().autofmt_xdate()                           # 날짜 글자를 비스듬히 눕혀 겹치지 않게
plt.ylabel("승객(천 명)"); plt.legend(ncol=2); plt.show()

**[결과 읽기]** 한 번 시험하면 홀트-윈터스가 이기지만, 여섯 번 시험하니 1등이 셋으로 갈리고 둘의 평균이 가장 좋았습니다(3.6과 같은 결론).

> **[분석 기법] 결합 예측: 둘의 평균이 하나보다 낫다**
>
> 서로 다르게 틀리는 두 예측을 평균하면 한쪽이 높게, 다른 쪽이 낮게 틀린 날 오차가 서로 깎여 크게 지는 해가 줄어듭니다. 10장의 랜덤 포레스트가 나무 수백 그루의 평균인 것도 같은 원리입니다(**앙상블**). 단, 둘이 똑같이 틀리면 평균해도 나아지지 않습니다.

### 한국 전력 파일 받기

코랩 파일 창에 파일을 올려 두었다면 이 셀은 건너뜁니다. 없으면 책 저장소(ch12/data)에서 받아 옵니다.

In [ ]:
import os, urllib.parse, urllib.request
KPX = "kpx_시간별_전국_전력수요량_2021_2025.csv"
if not os.path.exists(KPX):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch12/data/" + urllib.parse.quote(KPX), KPX)
print(os.path.getsize(KPX), "bytes")

## [파이썬 실습 12-5 ★] 한국 전력 데이터 불러오고 가장 높은 날 확인하기

한국전력거래소의 시간별 전력 수요 파일을 읽어 신원을 확인하고, 하루 단위(GWh)로 바꾼 뒤 가장 많이 쓴 날과 가장 적게 쓴 날을 찾으시오(03절 3.1, 3.4).

**① 읽고 신원 확인하기**

In [ ]:
DATA = ""                                           # 파일을 둔 폴더(코랩 파일 창에 올렸으면 빈 글자)
kp = pd.read_csv(DATA + "kpx_시간별_전국_전력수요량_2021_2025.csv")
print(kp.shape, "/ 빈칸이 있는 줄:", kp.isna().any(axis=1).sum())
kp = kp.dropna(subset=["날짜"])                      # 날짜까지 빈 줄은 버린다
kp["날짜"] = pd.to_datetime(kp["날짜"])
kp = kp.set_index("날짜").asfreq("D")                # 날짜를 인덱스로, 빠진 날은 빈 줄로 드러낸다
print("같은 날짜:", kp.index.duplicated().sum(), "/ 날짜 순서대로:", kp.index.is_monotonic_increasing,
      "/ 남은 빈칸:", kp.isna().sum().sum())

- 하루가 한 줄(날짜, 1시~24시의 MW)입니다. `isna( ).any(axis=1)`은 줄마다 "빈칸이 하나라도 있나"를 묻고, 빈 줄을 버린 뒤 `asfreq("D")`로 다시 보면 빠진 날이 없습니다.

> **[분석 기법] 데이터 신원 확인 네 가지**
>
> 어떤 데이터든 분석 전에 ① 크기와 기간 `shape`, ② 빈칸 `isna( ).sum( )`, ③ 같은 줄 `index.duplicated( ).sum( )`, ④ 순서 `index.is_monotonic_increasing`을 확인합니다. 빈칸은 왜 비었는지부터 묻습니다. 이 파일의 빈 줄은 여러 해의 파일을 이어 붙일 때 끝에 딸려 온 줄이라 버렸습니다. 값이 있어야 할 날이 비었다면 버리지 말고 그 까닭을 먼저 찾습니다.


**② 하루 GWh로 바꾸고, 가장 많이 쓴 날과 적게 쓴 날 찾기**

In [ ]:
c = kp.mean(axis=1) * 24 / 1000                     # 하루 GWh = 24시간 평균(MW) × 24 ÷ 1000
print(len(c), "일 / 하루 평균", round(c.mean(), 1), "GWh")
print("해마다 평균:", c.groupby(c.index.year).mean().round(1).to_dict())
print("가장 많은 날\n" + c.nlargest(3).round(1).to_string(header=False))
print("가장 적은 날\n" + c.nsmallest(3).round(1).to_string(header=False))
d = c.idxmax()
print("그날 가장 많이 쓴 시각:", kp.loc[d].idxmax(), int(kp.loc[d].max()), "MW")

- `mean(axis=1)`은 24개 열을 옆으로 평균합니다. 평균 MW(그 순간 쓰는 힘)에 24시간을 곱하면 하루의 MWh, 1,000으로 나누면 GWh입니다. `nlargest(3)`은 가장 큰 세 값을 날짜와 함께 돌려줍니다.

> **[분석 기법] 튀는 값은 지우기 전에 확인한다**
>
> 가장 크거나 작은 값 몇 개를 날짜와 함께 꺼내(`nlargest`, `nsmallest`) 바깥의 사실과 맞춰 봅니다. 달력(명절, 휴일), 날씨(폭염, 한파), 뉴스(행사, 사고)로 설명되면 **사건**이니 남기고, 설명되지 않으면 단위, 입력 실수, 집계 기준 변경 같은 **오류**를 의심합니다. 매출 기록의 최고 매출일이 블랙 프라이데이인지, 0이 하나 더 붙은 입력인지 가리는 것과 같습니다. 사건을 오류로 알고 지우면 모델은 폭염과 명절을 배울 기회를 잃습니다.


**③ 다섯 해 그래프에 두 날 표시하기**

In [ ]:
c.plot(color="#8EC5E8", lw=0.6, figsize=(7, 3), label="하루 수요")
c.rolling(28, center=True).mean().plot(color="#4466AE", label="4주 이동평균")   # 요일 톱니를 지운다
plt.scatter([c.idxmax(), c.idxmin()], [c.max(), c.min()], color="#E04B3C", zorder=3)
plt.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1))
plt.ylabel("GWh"); plt.xlabel(""); plt.show()

**[결과 읽기]** 가장 많은 날은 폭염과 한파, 가장 적은 날은 명절 연휴라 오류가 아니라 사건입니다(3.4). 해마다 평균은 1,530~1,574 GWh로 계단처럼 뛴 해가 없습니다.


## [파이썬 실습 12-6 ★] 요일, 달, 자기상관

요일별, 달별 평균 수요와 여러 시차의 자기상관을 구하시오(03절 3.2, 01절 [표 12-1]).

**① 요일별, 달별 평균**

In [ ]:
dow = c.groupby(c.index.dayofweek).mean()          # 0=월요일 … 6=일요일
print("요일 평균(월~일):", dow.round(1).tolist())
print("주말은 평일보다", round((1 - dow[[5, 6]].mean() / dow[:5].mean()) * 100, 1), "% 적다")
mon = c.groupby(c.index.month).mean()
print("가장 많은 달", mon.idxmax(), "월", round(mon.max(), 1), "/ 가장 적은 달", mon.idxmin(), "월", round(mon.min(), 1))

- `dayofweek`: 월요일이 0, 일요일이 6입니다. 그래서 `dow[[5, 6]]`은 주말, `dow[:5]`는 평일입니다.

> **[분석 기법] 묶어서 요약하는 세 가지 방법**
>
> `groupby( )`는 같은 이름끼리(요일, 해, 매장), `resample( )`은 시간 간격으로(주 `"W"`, 달 `"MS"`), `rolling( )`은 창을 한 칸씩 밀며(최근 7일) 묶습니다. 셋 모두 뒤에 `.mean( )`, `.sum( )`, `.max( )`를 붙여 요약합니다.


**② 시차별 자기상관**

In [ ]:
lags = [1, 2, 3, 4, 5, 6, 7, 14, 364, 365]
r = pd.Series({k: c.autocorr(k) for k in lags})   # k칸 민 자기 자신과의 상관계수
print("자기상관:", r.round(3).to_dict())

- `c.autocorr(k)`: c와 k칸 민 c의 상관계수입니다(식 12-3).
- `{k: … for k in lags}`: 시차마다 값을 구해 사전에 담는 한 줄짜리 반복입니다.

**③ 자기상관 그래프**

In [ ]:
r.plot.bar(color=["#E04B3C" if k in (7, 364) else "#8EC5E8" for k in r.index],
           figsize=(7, 3), rot=0)
plt.xlabel("시차(일)"); plt.ylabel("자기상관"); plt.show()

**[결과 읽기]** 시차 6·7이 다시 오르고 2~4가 낮은 것은 한 주의 모양 때문이고, 365일이 364일보다 낮은 것은 요일이 하루씩 어긋나기 때문입니다. 그런데 한국은 시차 1(0.765)과 7(0.802)의 차이가 작습니다. 더위와 추위가 며칠씩 이어져 어제도 오늘을 많이 닮기 때문입니다. 그래서 기준선 후보는 "어제", "지난주 같은 요일", "작년 같은 요일(364일 전)" 셋이고, 어느 쪽이 나은지는 3.3과 3.7에서 시험으로 가립니다.


## [파이썬 실습 12-7 ★] 2025년을 기준선과 프로펫으로 예측하기

2024년까지로 배우고 2025년 365일을 예측하시오. 작년 같은 요일과 두 가지 프로펫을 견주고, 가장 크게 틀린 날의 예측을 부분별로 나누어 보시오(03절 3.3, 3.5).

**① 정직한 "작년 같은 요일" 만들기**

In [ ]:
tr_e, te_e = c.loc[:"2024"], c.loc["2025"]          # 2024년까지 배우고 2025년 시험
f_364 = np.tile(tr_e.iloc[-364:].values, 2)[:365]   # 2024년의 마지막 52주를 두 번 잇는다
leak = c.shift(364).loc["2025"]                     # 흔한 실수: 시험 해의 값이 섞인다
print("평균으로 찍기 MAE", mae(te_e, tr_e.loc["2024"].mean()))
print("작년 같은 요일 MAE", mae(te_e, f_364), " (shift(364)로 하면", mae(te_e, leak), ")")

- `iloc[-364:]`: 학습 기간의 마지막 364일(52주)이라 요일이 맞습니다. 이것을 두 번 잇고 365일만 씁니다.
- `shift(364)`로 만들면 2025년 12월 31일 자리에 2025년 1월 1일 값이 들어갑니다. 2024년 말에는 알 수 없는 값입니다.

> **[자주 하는 실수] 데이터 누수: 시험 답이 학습에 섞인다**
>
> **데이터 누수**(leakage)는 예측하는 순간에는 알 수 없는 정보가 섞이는 것입니다. 흔한 꼴은 셋입니다. ① `shift( )`가 시험 기간의 값을 끌어옴(이 실습의 `shift(364)`), ② 전체 데이터로 표준화나 빈칸 채우기를 한 뒤 나눔(학습 데이터로만 `fit`), ③ 결과가 난 뒤에 생기는 열을 입력으로 씀(연체 예측에 "독촉 문자 횟수"). 점수가 너무 좋으면 먼저 누수를 의심합니다.


**② 프로펫 함수 만들기**

In [ ]:
def fit_prophet(series, holidays_on, days=365):
    df = series.reset_index()
    df.columns = ["ds", "y"]                        # 프로펫은 열 이름이 ds, y여야 한다
    m = Prophet(daily_seasonality=False)
    if holidays_on:
        m.add_country_holidays(country_name="KR")   # 한국 공휴일 달력을 알려 준다
    m.fit(df)
    return m, m.predict(m.make_future_dataframe(periods=days)).set_index("ds")

- `reset_index( )`: 날짜 인덱스를 열로 꺼냅니다. 프로펫은 날짜를 열로 받습니다.
- `make_future_dataframe( )`: 학습 기간 뒤 365일의 날짜 표를 만들고, `predict( )`가 예측값 `yhat`과 구간 `yhat_lower`, `yhat_upper`를 채웁니다.

**③ 두 번 맞추고 채점하고, 가장 크게 틀린 날 나누어 보기**

In [ ]:
m0, fc0 = fit_prophet(tr_e, False)
m1, fc1 = fit_prophet(tr_e, True)
p0, p1 = fc0["yhat"].loc["2025"], fc1["yhat"].loc["2025"]
print("프로펫 기본 MAE", mae(te_e, p0), "/ 프로펫 + 휴일 MAE", mae(te_e, p1))
err = (te_e - p1).abs().sort_values(ascending=False)   # 틀린 크기를 큰 것부터
print("크게 틀린 날:", [str(d.date()) for d in err.index[:3]])
d0 = err.index[0]
print(d0.date(), "실제", round(te_e[d0], 1), "/ 예측의 부분들:",
      fc1.loc[d0, ["trend", "weekly", "yearly", "holidays", "yhat"]].round(1).to_dict())

- 프로펫의 예측 `yhat`은 추세(`trend`) + 요일(`weekly`) + 한 해의 계절(`yearly`) + 휴일(`holidays`)의 합입니다. 크게 틀린 날은 이 부분들을 꺼내 보면 어느 부분이 잘못했는지 보입니다.

> **[분석 기법] 가장 크게 틀린 것부터 본다**
>
> 평균 오차 하나로는 어디서 틀리는지 모릅니다. 오차 크기를 `abs( )`로 구해 `sort_values(ascending=False)`로 늘어놓고 위에서 몇 개를 직접 들여다봅니다(**오차 분석**). 여기서는 위의 날들이 모두 2025년 추석 연휴에 몰려 있었고, 답은 "효과를 더하기만 하는 모델은 겹친 날을 모른다"였습니다. 집값 모델이면 크게 틀린 집이 신축인지 보는 식으로, 다음에 고칠 곳은 대개 여기서 나옵니다.


**④ 10월 한 달 그래프**

In [ ]:
oct25 = slice("2025-10-01", "2025-10-31")
te_e[oct25].plot(color="black", label="실제", figsize=(7, 3))
p1[oct25].plot(color="#E04B3C", label="프로펫 + 휴일")
sn25 = pd.Series(f_364, te_e.index)                # 배열에 2025년 날짜를 붙인다
sn25[oct25].plot(color="#8EC5E8", label="작년 같은 요일")
plt.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1))  # 범례는 그래프 위에
plt.ylabel("GWh"); plt.xlabel(""); plt.show()

- 시리즈의 `.plot( )`은 날짜 인덱스를 가로축으로 써서 날짜 글자가 겹치지 않게 그립니다. `bbox_to_anchor=(0.5, 1)`은 범례를 그래프 위 바깥에 둡니다.



**[결과 읽기]** [표 12-2]의 숫자가 그대로 나옵니다. 10월 5일은 일요일 효과와 추석 전날 효과를 둘 다 빼서 지나치게 낮게 예측했습니다(3.5).


## [파이썬 실습 12-8 ★] 해마다 기준점을 옮겨 전력 수요 예측하기

2023~2025년을 해마다 그 전해까지로 배워 예측하고, 작년 같은 요일, 프로펫, 둘의 평균을 견주시오(03절 3.6). 프로펫을 세 번 맞추므로 1분쯤 걸립니다.

**① 기준점을 한 해씩 옮기며 예측하기**

In [ ]:
rows = []
for Y in range(2023, 2026):                         # 2023~2025년을 차례로 시험
    tr, te = c.loc[:str(Y - 1)], c.loc[str(Y)]      # 그 전해까지 학습, 그해 시험
    sn = np.tile(tr.iloc[-364:].values, 2)[:len(te)]  # 작년 같은 요일 (시험 해 값은 쓰지 않음)
    p = fit_prophet(tr, True, len(te))[1]["yhat"].loc[str(Y)].values
    rows.append({"시험 해": Y, "배운 해 수": Y - 2021, "작년 같은 요일": mae(te, sn),
                 "프로펫+휴일": mae(te, p), "둘의 평균": mae(te, (sn + p) / 2)})

- 12-4와 같은 틀에 데이터와 모델만 바뀌었습니다. `len(te)`는 윤년(366일)을 챙기고, 결합 예측은 두 예측을 더해 2로 나누기만 하면 됩니다.

**② 성적표로 모으기**

In [ ]:
eroll = pd.DataFrame(rows).set_index("시험 해")      # 사전의 목록 → 표
print(eroll.to_string())
print(eroll.iloc[:, 1:].agg(["mean", "max"]).round(1).to_string())

- `iloc[:, 1:]`은 점수 열만, `agg(["mean", "max"])`는 평균과 가장 나빴던 해를 구합니다.

**③ 해마다 성적 그래프**

In [ ]:
eroll.iloc[:, 1:].plot.bar(color=["#8EC5E8", "#E04B3C", "#4466AE"], figsize=(7, 3), rot=0)
plt.ylabel("MAE(GWh)"); plt.xlabel("시험 해"); plt.show()

**[결과 읽기]** 해마다 값은 [표 12-3]과 같습니다. 프로펫은 배운 해가 셋, 넷으로 늘자 기준선을 이겼지만, 두 해로만 배운 2023년에는 크게 졌습니다. 평균은 둘의 평균이 가장 작고, 가장 나빴던 해도 다른 둘의 최악보다 낫습니다.


## [파이썬 실습 12-9 ★] 지연 특징으로 다음 날 예측하기

지연 특징과 달력 특징으로 랜덤 포레스트를 배우게 하고, 2025년의 매일 다음 날을 예측하시오(03절 3.7).

**① 휴일 열 만들기**

In [ ]:
kr = holidays.KR(years=range(2021, 2026))          # 한국 공휴일(설·추석 연휴, 대체 휴일 포함)
hol = pd.Series([d in kr for d in c.index.date], index=c.index).astype(int)

- 날짜마다 "공휴일 달력에 있나?"를 물어 참·거짓을 얻고, `astype(int)`로 1·0으로 바꿉니다(8장의 더미 변수).

**② 지연 특징 표 만들기**

In [ ]:
X = pd.DataFrame({"어제": c.shift(1), "지난주 같은 요일": c.shift(7),
                  "작년 같은 요일": c.shift(364), "지난 7일 평균": c.shift(1).rolling(7).mean(),
                  "달": c.index.month, "요일": c.index.dayofweek,
                  "휴일": hol, "어제가 휴일": hol.shift(1, fill_value=0)})
data = X.join(c.rename("y")).dropna()               # 정답 열을 붙이고 빈칸 줄을 버린다

- `shift(1)`, `shift(7)`, `shift(364)`가 어제, 지난주, 작년의 값을 오늘 줄로 끌어옵니다. 평균은 `shift(1)` 뒤에 구해 오늘 값이 섞이지 않게 했습니다.
- 날짜 하나에서 달력 특징(달, 요일, 휴일)과 지연 특징을 뽑아 열로 풀어 주는 것이 **특징 공학**입니다.

**③ 나누고, 기준선과 랜덤 포레스트 채점하기**

In [ ]:
trB, teB = data.loc[:"2024"], data.loc["2025"]
feats = [k for k in data.columns if k != "y"]       # 정답(y)을 뺀 나머지가 입력
print("어제 값 MAE", mae(teB.y, teB["어제"]), "/ 지난주 같은 요일 MAE", mae(teB.y, teB["지난주 같은 요일"]))
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=0, n_jobs=-1)
rf.fit(trB[feats], trB.y)                           # 2024년까지로 배운다
pred = rf.predict(teB[feats])                       # 2025년 매일의 다음 날
print("랜덤 포레스트 MAE", mae(teB.y, pred), "/ 하루 수요의", round(mae(teB.y, pred) / teB.y.mean() * 100, 1), "% / 배운 날", len(trB))

- 지연 특징 열이 그대로 기준선 예측이 됩니다. `teB["어제"]`가 곧 "어제 값" 예측입니다.

> **[잠깐만] 모델을 만들 때 정하는 값: 하이퍼파라미터**
>
> 학습으로 정해지지 않고 사람이 미리 정하는 값을 **하이퍼파라미터**라 합니다. `n_estimators`는 나무 수(많을수록 안정, 대신 느림), `min_samples_leaf`는 잎에 남길 최소 날 수(크면 덜 외움)입니다. `random_state=0`은 무작위의 출발점을 고정해 누가 돌려도 같은 결과를 내고(**재현성**), `n_jobs=-1`은 모든 코어를 씁니다. 고를 때는 시험 데이터가 아니라 학습 기간 안의 검증으로 고릅니다.


**④ 특징 중요도 그래프**

In [ ]:
imp = pd.Series(rf.feature_importances_, feats).sort_values()   # 기여한 몫
imp.plot.barh(color="#E04B3C", figsize=(6, 3)); plt.xlabel("중요도"); plt.show()

**[결과 읽기]** 랜덤 포레스트(43.1 GWh)는 두 기준선의 절반보다 작게 틀립니다. 어제 값은 월요일에, 지난주 값은 주중에 약한데(01절), 모델은 둘을 함께 보고 요일과 휴일로 어느 쪽을 믿을지 고릅니다.

> **[분석 기법] 특징 중요도를 읽을 때 주의할 점**
>
> `feature_importances_`는 나무가 갈라질 때 특징마다 오차를 줄인 몫입니다. 첫째, 닮은 특징끼리는 몫을 나눠 갖습니다. 둘째, 중요도는 원인이 아닙니다. 지난주 값이 전기를 쓰는 것이 아니라 사람들의 일과를 대신 알려 줄 뿐입니다. 시험 데이터에서 특징을 하나씩 섞어 보는 `permutation_importance( )`가 더 믿을 만합니다.



> **핵심 3줄**
>
> 날짜를 인덱스로 읽고, `groupby`·`resample`·`rolling`으로 묶고, `shift`·`autocorr`로 밀고 잽니다.
>
> 날짜로 나누고, 같은 `mae( )`로 기준선과 모델을 채점하며, 기준점을 옮겨 되풀이합니다.
>
> 특징은 그때 알던 값(`shift(1)` 이상)으로만 만들고, 가장 크게 틀린 것부터 들여다봅니다.

# 04절 예제를 파이썬으로 확인

책 04절의 [예제 12-1]~[예제 12-4]에서 손으로 계산한 값을 파이썬으로 다시 구합니다.

## [예제 12-1] 3개월 이동평균 = 무게와 값의 내적

In [ ]:
w = np.array([1, 1, 1]) / 3                         # 무게(커널) 세 개, 합은 1
print(round(np.array([112, 118, 132]) @ w, 1))      # 내적 = 3개월 이동평균

## [예제 12-2] 엇갈리는 시계열의 자기상관

In [ ]:
s = pd.Series([10, 20, 10, 20, 10])
print("시차 1:", round(s.autocorr(1), 3), " 시차 2:", round(s.autocorr(2), 3))

## [예제 12-3] 지수평활을 손으로 계산하기

In [ ]:
s_t = 112                                           # 첫 평활값
for v in [118, 132]:
    s_t = 0.3 * v + 0.7 * s_t                       # (식 12-4)
    print(round(s_t, 2))

## [예제 12-4] 작년 같은 달과 항공사 모델의 오차 견주기

In [ ]:
real = np.array([360, 342, 406])
for name, f in [("작년 같은 달", [340, 318, 362]), ("항공사 모델", [348.6, 331.6, 383.2])]:
    e = np.abs(real - np.array(f))
    print(f"{name}  MAE {e.mean():.1f}  MAPE {(e / real).mean() * 100:.1f}%")

# [미니 프로젝트 12] 독일 전력으로, 한국 데이터에서 못 본 것 겪어 보기

**질문.** 한국 데이터는 다섯 해, 빈칸이 없고, 재는 방법도 한 번도 바뀌지 않은 깨끗한 데이터였습니다. 실제 데이터는 그렇지 않을 때가 많습니다. 독일의 열두 해 전력 수요로 한국 데이터에서는 겪을 수 없었던 네 가지, 곧 측정을 늦게 시작한 열, 계단처럼 뛴 해, 처음 오는 휴일, 긴 기록을 겪어 보시오.

**데이터.** 독일 전체의 하루 전력 소비량(GWh) 2006~2017년 4,383일로, 송전망에서 잰 부하를 공개 데이터 프로젝트 OPSD가 정리한 것입니다. 풍력, 태양광 발전량 열도 있습니다. 인터넷 주소에서 바로 읽습니다.

**1단계: 늦게 시작한 열**([파이썬 실습 12-5] 응용). 열마다 빈칸 수와 첫 값이 있는 날(`first_valid_index( )`)을 구하고, 빈칸을 채울지 열을 버릴지 기간을 자를지 정하시오.

In [ ]:
URL_EL = "https://raw.githubusercontent.com/jenfly/opsd/master/opsd_germany_daily.csv"
el = pd.read_csv(URL_EL, parse_dates=["Date"], index_col="Date").asfreq("D")
g = el["Consumption"]                                # 하루 전력 소비, 단위 GWh
print(el.shape, "빈칸:", el.isna().sum().to_dict())
print("첫 값이 있는 날:", {k: str(el[k].first_valid_index().date()) for k in el.columns})

**2단계: 계단 찾기**([파이썬 실습 12-5] 응용). 해마다 평균의 변화율(`pct_change( )`)과 2014년의 달마다 전년 같은 달 대비 변화를 구하고, 세상이 바뀐 것인지 재는 방법이 바뀐 것인지 판단하시오.

In [ ]:
gy = g.groupby(g.index.year).mean()                  # 해마다 하루 평균
print("전년 대비 변화(%):", (gy.pct_change() * 100).round(1).loc[2009:2015].to_dict())
gm = g.resample("MS").mean()                         # 달마다 평균
ratio = (gm.loc["2014"].values / gm.loc["2013"].values - 1) * 100
print("2014년 달마다 전년 같은 달 대비(%):", np.round(ratio, 1))
gy.plot.bar(color=["#E04B3C" if t == 2014 else "#8EC5E8" for t in gy.index], figsize=(7, 3), rot=0)
plt.ylim(1200, 1420); plt.ylabel("하루 평균 수요(GWh)"); plt.xlabel(""); plt.show()

**3단계: 처음 오는 휴일**([파이썬 실습 12-7] 응용). 2016년까지로 배워 2017년을 예측하는 프로펫에 독일 공휴일(`country_name="DE"`)을 넣고, 가장 크게 틀린 날을 찾으시오. 그날이 학습 기간에 휴일이었던 적이 있는지 확인하고, 나라 달력 대신 평일에 든 공휴일을 모두 "평일 공휴일"이라는 한 이름으로 묶어 알려 주면 어떻게 되는지 보시오.(`Prophet(holidays=표)`, 표는 holiday와 ds 두 열).

In [ ]:
def fit_de(series, extra=None, country=True, days=365):
    df = series.reset_index(); df.columns = ["ds", "y"]
    m = Prophet(daily_seasonality=False, holidays=extra)
    if country:
        m.add_country_holidays(country_name="DE")        # 독일 공휴일 달력
    m.fit(df)
    return m, m.predict(m.make_future_dataframe(periods=days)).set_index("ds")["yhat"].loc["2017"]
tr_g, te_g = g.loc[:"2016"], g.loc["2017"]
m1, q1 = fit_de(tr_g)
d0 = (te_g - q1).abs().idxmax()                      # 가장 크게 틀린 날
print("프로펫 + 휴일 MAE", mae(te_g, q1), "/ 가장 크게 틀린 날", d0.date(), "실제", round(te_g[d0], 1), "예측", round(q1[d0], 1))
print("배운 휴일 수:", len(m1.train_holiday_names), "/ 종교개혁 기념일 있나:", "Reformation Day" in list(m1.train_holiday_names))
days = [d for d in holidays.Germany(years=range(2006, 2018)) if d.weekday() < 5]   # 평일에 든 공휴일만
extra = pd.DataFrame({"holiday": "평일 공휴일", "ds": pd.to_datetime(sorted(days))})  # 모두 한 이름으로
m2, q2 = fit_de(tr_g, extra, country=False)          # 나라 달력 대신 이 표를 알려 준다
print("평일 공휴일 무리 MAE", mae(te_g, q2), "/ 10월 31일 예측", round(q2["2017-10-31"], 1))

**4단계: 긴 기록의 힘**([파이썬 실습 12-8] 응용). 2017년을 예측하는 프로펫의 학습 시작 해를 2015, 2013, 2011, 2006년으로 바꾸어 MAE를 견주시오. 한국에서 두 해로만 배운 2023년의 결과([표 12-3])와 견주어 설명합니다.

In [ ]:
for start in ["2015", "2013", "2011", "2006"]:       # 학습 시작 해를 바꾼다(끝은 늘 2016년)
    print(start, "년부터 배움: MAE", mae(te_g, fit_de(g.loc[start:"2016"])[1]))

**5단계: 한국과 견주어 결론 쓰기.** 시차 1, 7일 자기상관과 2017년 다음 날 예측의 두 기준선(어제 값, 지난주 같은 요일)을 구해 한국과 견주고, 독일 데이터에서 배운 점을 [표 12-4]처럼 표로 쓰시오.

In [ ]:
print("자기상관 독일:", {k: float(round(g.autocorr(k), 3)) for k in [1, 7]}, "/ 한국:", {k: float(round(c.autocorr(k), 3)) for k in [1, 7]})
print("2017년 다음 날 MAE: 어제", mae(te_g, g.shift(1).loc["2017"]), "/ 지난주 같은 요일", mae(te_g, g.shift(7).loc["2017"]))

**제출물.** 그림 2장(연도별 평균의 계단, 2017년 10월의 실제와 예측), 성적표 1개(학습 시작 해별 MAE와 "평일 공휴일" 전후 MAE), 결론 5문장 이내.

**[표 12-5] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 빈칸이 언제부터 사라지는지 확인하고, 열을 버리거나 기간을 자른 까닭을 밝힘 | 빈칸 수만 셈 | 빈칸을 평균으로 채움 |
| 계단 판단(20%) | 연도별 변화와 달별 변화를 함께 보고 재는 방법의 변화를 의심함 | 연도별 변화만 봄 | 계단을 수요 증가로 해석함 |
| 기법 적용(30%) | 처음 오는 휴일을 무리로 묶어 고치고, 학습 길이를 바꿔 시간 순서로 시험함 | 둘 가운데 하나만 함 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | 한국과 다른 점을 주기와 데이터 품질로 설명하고 한계를 표로 씀 | 한계를 한두 가지 씀 | 숫자만 나열함 |